# Phase 7 — Baseline Model Comparison
## Parkinson's Disease Prediction Using Typing Characteristics

**Primary Dataset:** Tappy Extended Keystroke Dataset, Mendeley Data Version 3 (DOI: 10.17632/z39mhdsynx.3)  
**Working Branch:** `praneeth`  
**Scope & Phase Boundary:** Strict Phase 7 baseline model exploration. Evaluates 6 prospective scikit-learn architectures on the frozen 477 training participants using 5-fold Stratified Cross-Validation.  
**Primary Metric:** Balanced Accuracy (weights Sensitivity and Specificity equally to penalize majority-class exploitation).  
**Test Set Quarantine:** The 120 held-out test participants are strictly quarantined and never loaded or evaluated.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configure paths relative to repository root
repo_root = Path.cwd().resolve()
if repo_root.name == 'notebooks':
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

from src.modelling.run_baseline_models import (
    get_models,
    evaluate_fold,
    positive_scores,
)

print(f"Repository root: {repo_root}")


## 1. Verification of Training Cohort and Clean Predictors

We verify that the input feature table contains the certified 153 typing-derived features with no infinite values, and that exactly 477 training participants are isolated.

In [ ]:
features_path = repo_root / 'data/processed/features/participant_features_raw.parquet'
train_ids_path = repo_root / 'data/processed/splits/train_ids.csv'

features = pd.read_parquet(features_path)
train_ids = pd.read_csv(train_ids_path)

print(f"Total features parquet shape: {features.shape}")
print(f"Frozen training IDs: {len(train_ids)}")

# Isolate training cohort
training_set = set(train_ids['participant_id'].astype(str))
train_df = features[features['participant_id'].astype(str).isin(training_set)].copy()
X_train = train_df.drop(columns=['participant_id', 'parkinsons_label'])
y_train = train_df['parkinsons_label'].astype(int)

print(f"Training matrix X: {X_train.shape}")
print(f"Training labels y: {y_train.value_counts().to_dict()} (PD: {y_train.mean():.1%})")
assert X_train.shape == (477, 153), f"Unexpected X_train shape: {X_train.shape}"
assert not np.isinf(X_train.to_numpy(dtype=float)).any(), "Infinite values detected in predictors!"


## 2. Prospective Model Architectures & Leakage-Free Preprocessing

In strict compliance with TRIPOD+AI and the Phase 3 protocol, all imputations (`SimpleImputer(strategy='median')`) and feature standardizations (`StandardScaler()`) are embedded inside scikit-learn `Pipeline` objects to prevent cross-validation leakage.

In [ ]:
models = get_models()
for name, pipe in models.items():
    steps = [step_name for step_name, _ in pipe.steps]
    print(f"{name:22s} -> Pipeline steps: {steps}")


## 3. Cross-Validation Results Summary

We inspect the 5-fold Stratified Cross-Validation results computed by `src/modelling/run_baseline_models.py`.

In [ ]:
summary_path = repo_root / 'reports/modelling/baseline_cv_summary.csv'
summary_df = pd.read_csv(summary_path)

display_cols = [
    'model',
    'balanced_accuracy_mean',
    'balanced_accuracy_std',
    'sensitivity_mean',
    'specificity_mean',
    'precision_mean',
    'accuracy_mean',
    'roc_auc_mean',
    'average_precision_mean',
]

print("=== 5-Fold Stratified Cross-Validation Performance (Sorted by Balanced Accuracy) ===")
summary_df[display_cols].to_string(index=False)


## 4. Visual Analysis of Baseline Estimators

We display the diagnostic visualizations generated from the cross-validation experiments.

In [ ]:
from IPython.display import Image, display

figures_dir = repo_root / 'reports/modelling/figures'
for fig_name in [
    'fig1_balanced_accuracy_comparison.png',
    'fig2_sensitivity_vs_specificity.png',
    'fig3_roc_auc_vs_balanced_acc.png',
    'fig4_rf_aggregate_confusion_matrix.png',
]:
    fig_path = figures_dir / fig_name
    if fig_path.is_file():
        print(f"Displaying {fig_name}:")
        display(Image(filename=str(fig_path)))


## 5. Model Selection & Prospective Conclusion

### Key Findings:
1. **Deception of Standard Accuracy Exposed:** The Dummy Prior baseline achieves **72.33% standard accuracy** by simply predicting the majority PD class, but achieves exactly **50.00% balanced accuracy** with **0.0% specificity**. Any model evaluated primarily on standard accuracy is vulnerable to trivial majority-class collapse.
2. **Baseline Champion:** **Random Forest** achieves the highest 5-fold cross-validated Balanced Accuracy (**0.5938 ± 0.0350**), highest ROC-AUC (**0.6664 ± 0.0181**), and highest PR-AUC (**0.8373 ± 0.0192**).
3. **Prospective Selection Rule:** The margin between Random Forest and the second-place Logistic Regression (**0.5532 ± 0.0421**) is **+0.0406** (+4.06%), far exceeding the protocol's tie-break threshold of $\pm 0.005$. Therefore, Random Forest is selected as the candidate model architecture for Phase 8.
4. **Holdout Test Set Isolation:** All 120 test participants remain fully quarantined in cold storage for final unblinded evaluation in Phase 10.